# ---------------------------------------------------
# 1) Normalize using TRAINING statistics only
# ---------------------------------------------------

In [ ]:
import numpy as np
import cvxpy as cp
import matplotlib.pyplot as plt


train_mean = np.mean(X_train, axis=0)
train_std = np.std(X_train, axis=0, ddof=0)

# avoid division by zero
train_std = np.where(train_std == 0, 1, train_std)

X_train_normalized = (X_train - train_mean) / train_std
X_test_normalized = (X_test - train_mean) / train_std

print("First feature mean:", train_mean[0])
print("First feature std:", train_std[0])
print("Last feature mean:", train_mean[-1])
print("Last feature std:", train_std[-1])


# ---------------------------------------------------
# 2) Train linear soft-margin SVM in primal form
# ---------------------------------------------------

In [ ]:

def trainSVM(X_train_normalized, y_train, C):
    """
    Solves:
        min_{w,b,xi} 0.5 * ||w||^2 + C * sum(xi_i)
    s.t.
        y_i (w^T x_i + b) >= 1 - xi_i
        xi_i >= 0
    """
    X = np.asarray(X_train_normalized, dtype=float)
    y = np.asarray(y_train, dtype=float).reshape(-1)

    n_samples, n_features = X.shape

    w = cp.Variable(n_features)
    b = cp.Variable()
    xi = cp.Variable(n_samples)

    objective = cp.Minimize(0.5 * cp.sum_squares(w) + C * cp.sum(xi))
    constraints = [
        cp.multiply(y, X @ w + b) >= 1 - xi,
        xi >= 0
    ]

    problem = cp.Problem(objective, constraints)
    problem.solve()

    return w.value, np.array([b.value]), xi.value


# ---------------------------------------------------
# 3) Evaluate SVM accuracy
# ---------------------------------------------------

In [ ]:

def evalSVM(X_eval, y_eval, w, b):
    X = np.asarray(X_eval, dtype=float)
    y = np.asarray(y_eval, dtype=float).reshape(-1)

    scores = X @ w + b
    y_pred = np.where(scores >= 0, 1, -1)

    accuracy = np.mean(y_pred == y)
    return accuracy

# ---------------------------------------------------
# 4) Check requested outputs for C = 1 and C = 0
# ---------------------------------------------------

In [ ]:


w_1, b_1, xi_1 = trainSVM(X_train_normalized, y_train, C=1)
print("\nFor C = 1")
print("First three weights:", w_1[:3])
print("Bias:", b_1)
print("First three slack variables:", xi_1[:3])

w_0, b_0, xi_0 = trainSVM(X_train_normalized, y_train, C=0)
print("\nFor C = 0")
print("First three weights:", w_0[:3])
print("Bias:", b_0)
print("First three slack variables:", xi_0[:3])

# ---------------------------------------------------
# 5) Sweep over all requested C values
#    C = a * 10^q, a in {1,3,6}, q in {-4,-3,-2,-1,0,1}
# ---------------------------------------------------

In [ ]:
C_values = []
for q in [-4, -3, -2, -1, 0, 1]:
    for a in [1, 3, 6]:
        C_values.append(a * (10 ** q))

C_values = sorted(C_values)

train_accuracies = []
test_accuracies = []

for C in C_values:
    w, b, xi = trainSVM(X_train_normalized, y_train, C)
    train_acc = evalSVM(X_train_normalized, y_train, w, b)
    test_acc = evalSVM(X_test_normalized, y_test, w, b)

    train_accuracies.append(train_acc)
    test_accuracies.append(test_acc)

    print(f"C = {C:<7} | Train Acc = {train_acc:.4f} | Test Acc = {test_acc:.4f}")


# ---------------------------------------------------
# 6) Report best train/test accuracy
# ---------------------------------------------------

In [ ]:
best_train_idx = int(np.argmax(train_accuracies))
best_test_idx = int(np.argmax(test_accuracies))

print("\nMaximum training accuracy:", round(train_accuracies[best_train_idx], 4),
      "at C =", C_values[best_train_idx])

print("Maximum test accuracy:", round(test_accuracies[best_test_idx], 4),
      "at C =", C_values[best_test_idx])



# ---------------------------------------------------
# 7) Plot Training Accuracy vs C
# ---------------------------------------------------

In [ ]:

plt.figure(figsize=(8, 5))
plt.plot(C_values, train_accuracies, marker='o')
plt.xscale('log')
plt.xlabel('C')
plt.ylabel('Training Accuracy')
plt.title('Training Accuracy vs C')
plt.grid(True)
plt.show()


# ---------------------------------------------------
# 8) Plot Test Accuracy vs C
# ---------------------------------------------------

In [ ]:

plt.figure(figsize=(8, 5))
plt.plot(C_values, test_accuracies, marker='o')
plt.xscale('log')
plt.xlabel('C')
plt.ylabel('Test Accuracy')
plt.title('Test Accuracy vs C')
plt.grid(True)
plt.show()